# Prepare the Data

Checklist:
1. Cleaning - missing values, outliers
2. Feature selection - what we dropped
3. Feature engineering - new/transformed features
4. Scaling

##### 1. **Cleaning:**
Like we said in exploration, missing values usually mean something here so we don't want to impute them. Missing `Primary Fur Color`, `Age` (and the `"?"` entries) and `Location` just get their own `"Missing"` category. Missing highlight color probably means the squirrel is solid colored, and missing `Litter` should really be `"None"`. For the numeric columns we just use the median. `Above Ground Sighter Measurement` was a mix of `"FALSE"` and numbers, so we parse it to feet (`"FALSE"` = 0) and flag the ones that were missing.

##### 2. **Feature selection:**
We go through what we kept and dropped in Feature Groups below.

##### 3. **Feature engineering:**
- `Hectare` -> `hectare_row` and `hectare_col`
- `Highlight Fur Color` -> `has_highlights`
- `Shift` -> `is_pm`
- `Litter` -> ordinal (None < Some < Abundant)
- vocalizations as 0/1 flags
- no interaction checked -> `Indifferent`
- `Other Activities` / `Other Interactions` -> anchor similarity scores

##### 4. **Scaling:**
We use `StandardScaler` on the continuous columns. The skewed counts and height get a `log1p` first since they have long tails. The flags and one-hot columns are already 0/1 so we leave those alone.

### Imports

In [1]:
%load_ext autoreload
%autoreload 2

import pickle

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler

# custom transformers live in a .py file so the pickle loads elsewhere
from custom_transformers import *

from sklearn import set_config
set_config(transform_output="pandas")

### Load the training data and separate the target

In [2]:
TARGET_COL = "is_diseased"

train_set = pd.read_parquet("train_set.parquet")
X = train_set.drop(columns=TARGET_COL)
y = train_set[TARGET_COL].copy()

X.shape, y.shape, f"positive rate: {y.mean():.2%}"

((2418, 45), (2418,), 'positive rate: 12.53%')

## Feature Groups

Columns not listed get dropped. 

##### Dropped
- **Unique Squirrel ID:** just an ID, already did its job.
- **Lat/Long:** same thing as X/Y.
- **X / Y:** we originally kept these for the smaller local clusters, but tuning in Training dropped them for all 3 models. Seems like hectare row/col is enough for location.
- **Date / Date_parsed:** didn't see any trend in exploration, and these exact census days won't show up again anyways.
- **Hectare:** replaced by row/col.
- **Combination of Primary and Highlight Color:** redundant now that we have primary + `has_highlights`.
- **Anonymized Sighter:** this is an observer ID, the model would probably just learn who reports sick squirrels.
- **Sighter Observed Weather Data:** we have the OpenMeteo weather instead.
- **Litter Notes:** 99% missing.

##### Kept
- **Hectare row/col:** gives the clean splits we talked about in exploration.
- **Fur color, Age, Location:** one-hot with missing as its own category.
- **Activity, vocal and interaction flags:** in exploration these all pointed to either normal behavior or nutosis, so we keep all of them. We had an `any_vocalization` flag too, but none of the tuned models used it so we took it out.
- **Weather:** some signal, and we should be able to get it for any future sighting.

##### REVISIT columns
The main question here is if an app sighting would even have these.
- **Color notes:** 94% missing. Dropped.
- **Specific Location:** 84% missing, and where the squirrel was sitting didn't show much signal in exploration. Dropped.
- **Hectare Conditions + Notes:** nutosis rates were pretty flat, and an app user probably won't fill these out. Dropped.
- **Other Animal Sightings:** this is repeated for the whole hectare, so it's really just location again. Dropped.
- **Litter:** kept, missing just means "None".
- **Number of Squirrels:** kept, seems like a decent proxy for how crowded/exposed the squirrels are.
- **Number of sighters / Total Time of Sighting:** this is more about how the census was run than the squirrel itself. We kept them for now, but if the app doesn't record these they should probably go.

In [3]:
# dropped
DROP_COLS = [
    "Unique Squirrel ID",          # just an ID
    "Lat/Long",                    # same as X/Y
    "Date",                        # raw int date
    "Date_parsed",                 # census days won't generalize
    "Hectare",                     # replaced by row/col
    "Combination of Primary and Highlight Color",  # redundant
    "Anonymized Sighter",          # observer ID
    "Sighter Observed Weather Data",  # have real weather instead
    "Litter Notes",                # almost all missing
    "Color notes",                 # REVISIT
    "Specific Location",           # REVISIT
    "Hectare Conditions",          # REVISIT
    "Hectare Conditions Notes",    # REVISIT
    "Other Animal Sightings",      # REVISIT
    "X", "Y",                      # tuning dropped these for all 3 models, hectare row/col covers location
]

# location
HECTARE_COL = ["Hectare"]          # -> row/col

# categorical
CATEGORICAL_COLS = ["Primary Fur Color", "Age", "Location"]   # missing = own category
HIGHLIGHT_COL = ["Highlight Fur Color"]      # -> has_highlights
SHIFT_COL = ["Shift"]                        # -> is_pm
LITTER_COL = ["Litter"]                      # REVISIT: None < Some < Abundant

# booleans
ACTIVITY_BOOLS = ["Running", "Chasing", "Climbing", "Eating", "Foraging", "Tail flags", "Tail twitches"]
VOCAL_BOOLS = ["Kuks", "Quaas", "Moans"]     # no any_vocalization, none of the tuned models used it
INTERACTION_BOOLS = ["Approaches", "Indifferent", "Runs from"]

# numeric
HEIGHT_COL = ["Above Ground Sighter Measurement"]
SKEWED_COUNT_COLS = [              # skewed -> log
    "Hectare Squirrel Number",
    "Number of Squirrels",       # REVISIT
    "Number of sighters",        # REVISIT
    "Total Time of Sighting",    # REVISIT
]
WEATHER_COLS = ["shift_temp_c", "shift_precip_mm", "shift_wind_kmh"]

# text anchors (starting guesses, refine with probe_anchor)
ACTIVITY_ANCHORS = ["sitting still", "chasing", "digging burying nut", "running", "playing"]
INTERACTION_ANCHORS = ["approaches people", "curious", "runs away", "dog"]   # dropped "looking at me" after Training

## Text Anchors

##### Other Activities
- **sitting still, chasing, playing:** these match pretty cleanly.
- **digging burying nut:** picks up "burying nut(s)" and "digging & burying", so it handles the different wordings well.
- **running:** mostly matches "running from #7" or "running from dog", so it's really just running away, which the checkboxes already cover.

##### Other Interactions
- **approaches people, curious, runs away, dog:** these all look like good matches.
- **looking at me:** starts drifting into "looking/begging for food", which seems more about food than nutosis.

Going forward, it seems like "running" and "looking at me" are the weakest anchors. We tried training with and without them, and all 3 tuned models dropped "looking at me", so we took it out. "running" was more mixed (2 of 3 dropped it), so we left it in and let Training keep deciding.

In [4]:
# top matching notes for each anchor
for phrase in ACTIVITY_ANCHORS:
    display(probe_anchor(train_set, "Other Activities", phrase, top_n=5))

,Other Activities,probe_0_sitting still
981,sitting,1.0
870,sitting,1.0
379,sitting,1.0
15,sitting,1.0
1417,sitting,1.0


,Other Activities,probe_0_chasing
672,"chasing (#8),scratching",1.000000
1454,"chasing (#8),playing?",1.000000
1219,"chasing (#7),playing?",1.000000
1593,"chasing (#4),digging",1.000000
430,chasing and being chased by #6,0.847633


,Other Activities,probe_0_digging burying nut
2163,burying nut,0.930374
90,burying nut,0.930374
1329,burying nut,0.930374
1491,burying nuts,0.899449
846,digging & burying,0.875729


,Other Activities,probe_0_running
756,running from #7,0.821504
1252,running from #3,0.810157
802,running (from a dog),0.770376
2246,running from dog,0.770376
1257,running (up tree),0.761129


,Other Activities,probe_0_playing
2118,playing,1.0
1443,playing,1.0
1962,playing,1.0
1234,playing,1.0
1848,playing?,1.0


In [5]:
# "looking at me" was dropped after Training, still probed here to show why
for phrase in INTERACTION_ANCHORS + ["looking at me"]:
    display(probe_anchor(train_set, "Other Interactions", phrase, top_n=5))

,Other Interactions,probe_0_approaches people
590,"approaches (me),runs from (me)",0.809532
175,"approaches (primarily),runs from (initially),f...",0.809532
200,"approaches (slightly),wary of humans",0.805436
926,approaches (very close),0.792930
326,approaches (came very close to me),0.779504


,Other Interactions,probe_0_curious
822,curious,1.0
2128,curious,1.0
464,curious,1.0
1699,curious,1.0
1747,curious,1.0


,Other Interactions,probe_0_runs away
2249,runs away from the ball,0.900692
263,runs from (us),0.863937
343,runs off,0.863937
1402,runs from (me),0.863937
316,runs from (us),0.863937


,Other Interactions,probe_0_dog
169,dog,1.000000
663,w/ dog,1.000000
394,dogs,0.883593
1983,scared by dog,0.863764
186,dog chased,0.843173


,Other Interactions,probe_0_looking at me
1114,looking,1.000000
918,looking at me,1.000000
121,looking/begging for food,0.707135
1064,"very curious,looked like it expected food",0.676667
1183,look at us and continue hiding the food,0.641968


## Build the Preprocessor

In [6]:
hectare_pipeline = Pipeline(steps=[
    ("grid", HectareGridTransformer()),
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("missing", MissingAsCategory(missing_tokens=("?", ""), fill_value="Missing")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

litter_pipeline = Pipeline(steps=[
    ("missing", MissingAsCategory(fill_value="None")),
    ("ordinal", OrdinalEncoder(
        categories=[["None", "Some", "Abundant"]],
        handle_unknown="use_encoded_value", unknown_value=-1,
    )),
])

height_pipeline = Pipeline(steps=[
    ("parse", SightingHeightTransformer()),
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),  # flag missing
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scale", StandardScaler()),
])

skewed_count_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scale", StandardScaler()),
])

numeric_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

preprocessor = ColumnTransformer(transformers=[
    ("drop", "drop", DROP_COLS),
    ("hectare", hectare_pipeline, HECTARE_COL),
    ("categorical", categorical_pipeline, CATEGORICAL_COLS),
    ("has_highlights", PresenceFlag(feature_name="has_highlights"), HIGHLIGHT_COL),
    ("is_pm", FunctionTransformer(shift_is_pm, feature_names_out=is_pm_name), SHIFT_COL),
    ("litter", litter_pipeline, LITTER_COL),
    ("activity", BooleanFlags(), ACTIVITY_BOOLS),
    ("vocal", BooleanFlags(), VOCAL_BOOLS),
    ("interaction", InteractionFlags(merge_none_into_indifferent=True), INTERACTION_BOOLS),
    ("height", height_pipeline, HEIGHT_COL),
    ("counts", skewed_count_pipeline, SKEWED_COUNT_COLS),
    ("weather", numeric_pipeline, WEATHER_COLS),
    ("activity_text", ConceptSimilarityTransformer(concepts=ACTIVITY_ANCHORS, prefix="act"), ["Other Activities"]),
    ("interaction_text", ConceptSimilarityTransformer(concepts=INTERACTION_ANCHORS, prefix="int"), ["Other Interactions"]),
], remainder="drop")

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('drop', ...), ('hectare', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feat

### Sanity check: every column is accounted for

In [7]:
# every column should be used or dropped on purpose
used = {col for _, _, cols in preprocessor.transformers for col in cols}
unaccounted = [c for c in X.columns if c not in used]
missing_from_data = [c for c in used if c not in X.columns]
print("Not in any group (silently dropped):", unaccounted)
print("Listed but not in data:", missing_from_data)

Not in any group (silently dropped): []
Listed but not in data: []


Looks like every column is either used or dropped on purpose.

## Fit and Transform

In [8]:
X_prepared = preprocessor.fit_transform(X, y)
X_prepared

,hectare__hectare_row,hectare__hectare_col,categorical__Primary Fur Color_Black,categorical__Primary Fur Color_Cinnamon,categorical__Primary Fur Color_Gray,categorical__Primary Fur Color_Missing,categorical__Age_Adult,categorical__Age_Juvenile,categorical__Age_Missing,categorical__Location_Above Ground,...,weather__shift_wind_kmh,activity_text__act_0_sitting still,activity_text__act_1_chasing,activity_text__act_2_digging burying nut,activity_text__act_3_running,activity_text__act_4_playing,interaction_text__int_0_approaches people,interaction_text__int_1_curious,interaction_text__int_2_runs away,interaction_text__int_3_dog
0,0.137767,-1.070630,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.036557,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-0.637656,-1.070630,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,...,-0.623739,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,-0.482571,0.144951,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,1.042723,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1.300902,1.360533,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,-0.700599,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.068275,0.550145,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,-0.372197,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2413,0.835648,-1.070630,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,-0.417615,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2414,-0.017317,-1.475824,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,0.211239,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2415,0.215310,-0.260242,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,1.322214,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2416,0.758106,-1.070630,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,-0.106681,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
# no NaNs, all numeric
print("shape:", X_prepared.shape)
print("NaNs remaining:", int(X_prepared.isna().sum().sum()))
print("non-numeric columns:", list(X_prepared.select_dtypes(exclude="number").columns))
X_prepared.describe().T

shape: (2418, 46)
NaNs remaining: 0
non-numeric columns: []


,count,mean,std,min,25%,50%,75%,max
hectare__hectare_row,2418.0,2.938556e-17,1.000207,-1.413079,-0.870283,-0.249944,0.990733,1.766156
hectare__hectare_col,2418.0,2.130453e-17,1.000207,-1.475824,-0.665436,0.144951,0.550145,1.765726
categorical__Primary Fur Color_Black,2418.0,3.515302e-02,0.184205,0.000000,0.000000,0.000000,0.000000,1.000000
categorical__Primary Fur Color_Cinnamon,2418.0,1.290323e-01,0.335305,0.000000,0.000000,0.000000,0.000000,1.000000
categorical__Primary Fur Color_Gray,2418.0,8.176179e-01,0.386239,0.000000,1.000000,1.000000,1.000000,1.000000
categorical__Primary Fur Color_Missing,2418.0,1.819686e-02,0.133690,0.000000,0.000000,0.000000,0.000000,1.000000
categorical__Age_Adult,2418.0,8.490488e-01,0.358075,0.000000,1.000000,1.000000,1.000000,1.000000
categorical__Age_Juvenile,2418.0,1.066998e-01,0.308795,0.000000,0.000000,0.000000,0.000000,1.000000
categorical__Age_Missing,2418.0,4.425145e-02,0.205696,0.000000,0.000000,0.000000,0.000000,1.000000
categorical__Location_Above Ground,2418.0,2.783292e-01,0.448269,0.000000,0.000000,0.000000,1.000000,1.000000


In [10]:
# test set: transform only, same columns
test_set = pd.read_parquet("test_set.parquet")
X_test_prepared = preprocessor.transform(test_set.drop(columns=TARGET_COL))
assert list(X_test_prepared.columns) == list(X_prepared.columns)
print("test shape:", X_test_prepared.shape, "| NaNs:", int(X_test_prepared.isna().sum().sum()))

test shape: (605, 46) | NaNs: 0


A few things to note from the output:
- `Moans` is almost always 0 (only 2 rows). We tried an `any_vocalization` flag for this but the models didn't use it.
- `Total Time of Sighting` has one really low outlier (min -10.5). Probably a sighting that was close to 0 minutes.
- `Number of sighters` and `shift_precip_mm` are mostly one value, most shifts had 1 sighter and no rain.
- The text scores go a little negative (-0.02), which seems normal for similarity scores.

No constant columns or NaNs, and the test set comes out with the same 46 columns, so it looks good to go.

## Save the Preprocessor

In [11]:
with open("preprocessor.pkl", "wb") as f:
    pickle.dump(preprocessor, f)

In [12]:
# reload check
with open("preprocessor.pkl", "rb") as f:
    reloaded = pickle.load(f)
pd.testing.assert_frame_equal(reloaded.transform(X), X_prepared)
print("preprocessor.pkl reloads correctly")

preprocessor.pkl reloads correctly


## Notes for Training

##### Preprocessing to tune in Training
- activity anchors with and without "running", interaction anchors with and without "dog"
- `merge_none_into_indifferent` on/off
- `any_vocalization` on/off (still checking even though we removed it)
- count imputer (median vs mean) and the height missing flag
- dropping hectare, weather, litter or `has_highlights`
- if `Number of sighters` / `Total Time of Sighting` don't end up mattering, we should drop them here

After the first round of tuning we dropped X/Y, `any_vocalization` and the "looking at me" anchor here, since all 3 models dropped them.

##### Class imbalance
We think this should be handled in the model instead of here. If we resampled in the preprocessor it would also mess with the test set and new sightings. So in Training we try different `class_weight` values (`None`, `"balanced"`, `{0: 1, 1: 5}`) and score with F2 since we care more about recall.